In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    LongType, DecimalType, BooleanType
)

spark = (
    SparkSession.builder
    .appName("OmniLake-Silver-BTCUSDT")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "64")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .getOrCreate()
)

bronze_path = "/home/belal/OmniLake-DEPI-Graduation-Project-/data_lake/bronze/BTCUSDT/BTCUSDT-trades-2024-01.csv"
silver_path = "/home/belal/OmniLake-DEPI-Graduation-Project-/data_lake/silver/BTCUSDT"

raw_schema = StructType([
    StructField("trade_id", LongType(), True),
    StructField("price", DecimalType(18, 8), True),
    StructField("quantity", DecimalType(18, 8), True),
    StructField("quote_quantity", DecimalType(18, 8), True),
    StructField("timestamp_ms", LongType(), True),
    StructField("is_buyer_maker", BooleanType(), True),
    StructField("is_best_match", BooleanType(), True),
])

df_raw = (
    spark.read
    .option("header", "false")
    .schema(raw_schema)
    .csv(bronze_path)
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/22 00:07:40 WARN Utils: Your hostname, LAPTOP-LRPBMQDJ, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/22 00:07:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/belal/OmniLake-DEPI-Graduation-Project-/venv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/22 00:07:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
import os

project_root = "/home/belal/OmniLake-DEPI-Graduation-Project-"
bronze_base = os.path.join(project_root, "data_lake", "bronze")

print("Notebook CWD:", os.getcwd())
print("Bronze base exists:", os.path.exists(bronze_base))
if os.path.exists(bronze_base):
    for root, dirs, files in os.walk(bronze_base):
        print("DIR:", root, "| FILES:", files[:5])

Notebook CWD: /home/belal/OmniLake-DEPI-Graduation-Project-/src
Bronze base exists: True
DIR: /home/belal/OmniLake-DEPI-Graduation-Project-/data_lake/bronze | FILES: []
DIR: /home/belal/OmniLake-DEPI-Graduation-Project-/data_lake/bronze/BTCUSDT | FILES: ['BTCUSDT-trades-2024-01.csv']
DIR: /home/belal/OmniLake-DEPI-Graduation-Project-/data_lake/bronze/BTCUSDT/.ipynb_checkpoints | FILES: []


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    LongType, DecimalType, BooleanType
)

spark = (
    SparkSession.builder
    .appName("OmniLake-Silver-BTCUSDT")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .getOrCreate()
)

bronze_path = "/home/belal/OmniLake-DEPI-Graduation-Project-/data_lake/bronze/BTCUSDT"

raw_schema = StructType([
    StructField("trade_id", LongType(), True),
    StructField("price", DecimalType(18, 8), True),
    StructField("quantity", DecimalType(18, 8), True),
    StructField("quote_quantity", DecimalType(18, 8), True),
    StructField("timestamp_ms", LongType(), True),
    StructField("is_buyer_maker", BooleanType(), True),
    StructField("is_best_match", BooleanType(), True),
])

df_raw = (
    spark.read
    .option("header", "false")
    .schema(raw_schema)
    .csv(bronze_path)
)

df_raw.printSchema()
df_raw.show(5, truncate=False)

root
 |-- trade_id: long (nullable = true)
 |-- price: decimal(18,8) (nullable = true)
 |-- quantity: decimal(18,8) (nullable = true)
 |-- quote_quantity: decimal(18,8) (nullable = true)
 |-- timestamp_ms: long (nullable = true)
 |-- is_buyer_maker: boolean (nullable = true)
 |-- is_best_match: boolean (nullable = true)

+----------+--------------+----------+--------------+-------------+--------------+-------------+
|trade_id  |price         |quantity  |quote_quantity|timestamp_ms |is_buyer_maker|is_best_match|
+----------+--------------+----------+--------------+-------------+--------------+-------------+
|3344774379|42283.58000000|0.00069000|29.17567020   |1704067200000|true          |true         |
|3344774380|42283.59000000|0.00144000|60.88836960   |1704067200001|false         |true         |
|3344774381|42283.58000000|0.00102000|43.12925160   |1704067200003|true          |true         |
|3344774382|42283.59000000|0.00219000|92.60106210   |1704067200003|false         |true         

In [3]:
silver_path = "../data_lake/silver/BTCUSDT"

df_silver = (
    df_raw
    .filter(
        F.col("trade_id").isNotNull() &
        F.col("timestamp_ms").isNotNull() &
        (F.col("price") > 0) &
        (F.col("quantity") > 0)
    )
    #   milliseconds إلى timestamp 
    .withColumn("trade_timestamp", (F.col("timestamp_ms") / 1000).cast("timestamp"))
    .withColumn("trade_date", F.to_date("trade_timestamp"))
    # quote_quantity 
    .withColumn(
        "quote_quantity",
        F.coalesce(
            F.col("quote_quantity"),
            (F.col("price") * F.col("quantity")).cast(DecimalType(18, 8))
        )
    )
    
    .withColumn("symbol", F.lit("BTCUSDT"))
    .dropDuplicates(["symbol", "trade_id"])
    #  rearranged columns for silver
    .select(
        "symbol",
        "trade_id",
        "trade_timestamp",
        "trade_date",
        "price",
        "quantity",
        "quote_quantity",
        "is_buyer_maker",
        "is_best_match"
    )
)

(
    df_silver.write
    .mode("overwrite")
    .partitionBy("trade_date")
    .parquet(silver_path)
)

print("Silver write completed successfully!")

Silver write completed successfully!


In [4]:
df_silver_check = spark.read.parquet("/home/belal/OmniLake-DEPI-Graduation-Project-/data_lake/silver/BTCUSDT")

print("Total Silver rows:", df_silver_check.count())
print("Distinct trade_dates:", df_silver_check.select("trade_date").distinct().count())
df_silver_check.printSchema()
df_silver_check.show(5, truncate=False)

Total Silver rows: 52549865


Distinct trade_dates: 32
root
 |-- symbol: string (nullable = true)
 |-- trade_id: long (nullable = true)
 |-- trade_timestamp: timestamp (nullable = true)
 |-- price: decimal(18,8) (nullable = true)
 |-- quantity: decimal(18,8) (nullable = true)
 |-- quote_quantity: decimal(18,8) (nullable = true)
 |-- is_buyer_maker: boolean (nullable = true)
 |-- is_best_match: boolean (nullable = true)
 |-- trade_date: date (nullable = true)

+-------+----------+-----------------------+--------------+----------+--------------+--------------+-------------+----------+
|symbol |trade_id  |trade_timestamp        |price         |quantity  |quote_quantity|is_buyer_maker|is_best_match|trade_date|
+-------+----------+-----------------------+--------------+----------+--------------+--------------+-------------+----------+
|BTCUSDT|3364474390|2024-01-11 00:00:00.382|45957.05000000|0.00044000|20.22110200   |true          |true         |2024-01-11|
|BTCUSDT|3364474442|2024-01-11 00:00:03.099|45958.46000000|0.0